# Iteris — Week 2 · Env Validation & Compute Benchmark

**Purpose:** Lock the `SegmentationEnv` contract before writing any agent code.  
Combines the three pre-coding deliverables from the council review into one notebook:

1. **State visualisation** — confirm 4 channels render correctly
2. **Reward sanity test** — handcrafted perturbations vs. expected reward sign
3. **Compute benchmark** — 1000 random steps on real T4, project to 50k / 100k

**Inputs needed (Kaggle):**
- `camus` dataset (same as Week 1)
- Week 1 outputs uploaded as a Kaggle Dataset, with: `camus_pred_masks/*.npy`, `camus_test_scores.csv`

**Decision points:** Go/no-go on training budget; lock state shape; lock reward formula.

## 0 · Imports

In [ ]:
%%capture
!pip install monai[all] -q

In [ ]:
import os, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as ndi

import torch
import monai
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    Orientationd, Spacingd, Resized, ScaleIntensityd,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| MONAI:', monai.__version__, '| Device:', DEVICE)

## 1 · Configuration

In [ ]:
CFG = dict(
    # ─── Data paths ───────────────────────────────────────────────────────
    camus_root      = '/kaggle/input/datasets/anfaalhossain/camus/CAMUS',
    week1_outputs   = '/kaggle/input/iteris-week1',     # ← upload Week 1 outputs as Kaggle Dataset
    masks_subdir    = 'camus_pred_masks',
    scores_csv      = 'camus_test_scores.csv',
    # ─── Env spec (LOCKED) ────────────────────────────────────────────────
    image_size      = (256, 256),
    num_classes     = 4,
    target_class    = 1,         # 1=LV_endo  2=LV_epi  3=LA  (sanity test uses LV_endo)
    class_names     = ['background', 'LV_endo', 'LV_epi', 'LA'],
    max_steps       = 20,
    shift_px        = 2,         # discrete shift magnitude (pixels)
    sdt_clip        = 20.0,      # SDT clipped to ±20 px then normalised to ±1
    early_stop_eps  = 0.001,     # |ΔDice| threshold for composite stop
    early_stop_hd   = 0.5,       # |ΔHD95| threshold (px)
    early_stop_n    = 3,         # consecutive steps below thresholds
    # ─── Benchmark ────────────────────────────────────────────────────────
    benchmark_steps = 1000,
    seed            = 42,
)

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
print(f'Target class for sanity / benchmark: {CFG["class_names"][CFG["target_class"]]}')

## 2 · Load Sample Data
Loads one test sample (image + GT + U-Net init mask) using the same MONAI pipeline as Week 1.  
Uses the per-patient CSV from Week 1 to find a sample that already has a `.npy` U-Net prediction.

In [ ]:
scores_df = pd.read_csv(Path(CFG['week1_outputs']) / CFG['scores_csv'])
print(f'Test samples available: {len(scores_df)}')
print(scores_df.head(3))

# Pick a mid-range Dice sample (not best, not worst) — better stress test
scores_df = scores_df.sort_values('dice_LV_endo').reset_index(drop=True)
sample = scores_df.iloc[len(scores_df) // 2]
print(f'\nSelected sample: {sample["patient"]}_{sample["view"]}_{sample["phase"]} '
      f'(LV_endo Dice = {sample["dice_LV_endo"]:.4f})')

In [ ]:
# Find image + label paths in the CAMUS dataset (Week 1 ingestion logic)
patient_dir = Path(CFG['camus_root']) / sample['patient']
if not patient_dir.exists():
    # auto-descend if dataset has wrapper folder
    for sub in Path(CFG['camus_root']).iterdir():
        if sub.is_dir():
            cand = sub / sample['patient']
            if cand.exists():
                patient_dir = cand
                break

EXTS = ['.nii.gz', '.nii', '.mhd']
img_path = lbl_path = None
for ext in EXTS:
    cand_img = patient_dir / f'{sample["patient"]}_{sample["view"]}_{sample["phase"]}{ext}'
    cand_lbl = patient_dir / f'{sample["patient"]}_{sample["view"]}_{sample["phase"]}_gt{ext}'
    if cand_img.exists():
        img_path, lbl_path = cand_img, cand_lbl
        break

init_path = (Path(CFG['week1_outputs']) / CFG['masks_subdir']
             / f'{sample["patient"]}_{sample["view"]}_{sample["phase"]}_pred.npy')

print(f'Image:  {img_path}')
print(f'Label:  {lbl_path}')
print(f'Init :  {init_path}')
assert img_path.exists() and lbl_path.exists() and init_path.exists()

# MONAI pipeline (same as Week 1)
load_tfm = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    Orientationd(keys=['image', 'label'], axcodes='RAS'),
    Spacingd(keys=['image','label'], pixdim=(1.0, 1.0, -1), mode=('bilinear', 'nearest')),
    Resized(keys=['image','label'], spatial_size=CFG['image_size'], mode=('bilinear', 'nearest')),
    ScaleIntensityd(keys=['image'], minv=0.0, maxv=1.0),
    EnsureTyped(keys=['image','label'], dtype=(torch.float32, torch.long)),
])
out      = load_tfm({'image': str(img_path), 'label': str(lbl_path)})
image    = out['image'][0].numpy().astype(np.float32)        # (H, W) in [0, 1]
gt_multi = out['label'][0].numpy().astype(np.int64)           # (H, W) ints 0..C-1
init_multi = np.load(init_path).astype(np.int64)              # (H, W) ints 0..C-1

# Binarise for the target class
tc       = CFG['target_class']
gt       = (gt_multi   == tc).astype(np.uint8)
init     = (init_multi == tc).astype(np.uint8)

print(f'\nimage  : {image.shape} [{image.min():.2f}, {image.max():.2f}]')
print(f'gt     : {gt.shape}  positives = {gt.sum()}')
print(f'init   : {init.shape}  positives = {init.sum()}')

## 3 · `SegmentationEnv` (locked v2 contract)

In [ ]:
STRUCT = ndi.generate_binary_structure(2, 1)  # 4-connectivity for dilate/erode

def dice_score(m1: np.ndarray, m2: np.ndarray, eps: float = 1e-6) -> float:
    inter = (m1 & m2).sum()
    return (2.0 * inter + eps) / (m1.sum() + m2.sum() + eps)

def hd95_px(m1: np.ndarray, m2: np.ndarray) -> float:
    """95th percentile Hausdorff distance in pixels. Returns 0 if both masks empty."""
    if not m1.any() or not m2.any():
        return 0.0 if (m1.any() == m2.any()) else float(max(m1.shape))
    edges_a = m1 ^ ndi.binary_erosion(m1, STRUCT)
    edges_b = m2 ^ ndi.binary_erosion(m2, STRUCT)
    if not edges_a.any() or not edges_b.any():
        return 0.0
    dt_b = ndi.distance_transform_edt(~edges_b)
    dt_a = ndi.distance_transform_edt(~edges_a)
    d_ab = dt_b[edges_a]
    d_ba = dt_a[edges_b]
    return float(np.percentile(np.concatenate([d_ab, d_ba]), 95))

def shifted(mask: np.ndarray, dy: int, dx: int) -> np.ndarray:
    """Translate mask by (dy, dx) px with zero-fill (no wrap-around)."""
    out = np.zeros_like(mask)
    H, W = mask.shape
    y_src = slice(max(0, -dy), H - max(0, dy))
    y_dst = slice(max(0, dy),  H - max(0, -dy))
    x_src = slice(max(0, -dx), W - max(0, dx))
    x_dst = slice(max(0, dx),  W - max(0, -dx))
    out[y_dst, x_dst] = mask[y_src, x_src]
    return out


class SegmentationEnv:
    """
    Per-class binary boundary-refinement environment (v2, locked).

    State (4, H, W) float32:
        ch 0 : image  ∈ [0, 1]
        ch 1 : current binary mask  ∈ {0, 1}
        ch 2 : signed distance transform of current mask, normalised to [-1, 1]
        ch 3 : U-Net init mask (fixed for the whole episode)

    Discrete actions (7):
        0 dilate   1 erode   2 ↑   3 ↓   4 ←   5 →   6 no-op

    Continuous actions (2): (dy, dx) ∈ [-0.1, 0.1] (fraction of image)

    Reward: r_t = Dice(mask_t, GT) - Dice(mask_{t-1}, GT)
    Done :  t ≥ max_steps  OR  composite stop (|ΔDice|<eps AND |ΔHD95|<eps for N steps)
    """

    NUM_DISCRETE_ACTIONS = 7

    def __init__(self, image, gt_mask, init_mask, max_steps=20,
                 action_type='discrete', shift_px=2, sdt_clip=20.0,
                 stop_eps_dice=0.001, stop_eps_hd=0.5, stop_n=3):
        assert action_type in ('discrete', 'continuous')
        assert image.shape == gt_mask.shape == init_mask.shape
        self.image     = image.astype(np.float32)
        self.gt        = gt_mask.astype(np.uint8)
        self.init_mask = init_mask.astype(np.uint8)
        self.H, self.W = image.shape
        self.max_steps = max_steps
        self.action_type = action_type
        self.shift_px  = shift_px
        self.sdt_clip  = sdt_clip
        self.stop_eps_dice = stop_eps_dice
        self.stop_eps_hd   = stop_eps_hd
        self.stop_n        = stop_n
        self.reset()

    def reset(self):
        self.mask    = self.init_mask.copy()
        self.t       = 0
        self.history = [self._snapshot_metrics()]   # list of (dice, hd95)
        return self._state()

    def step(self, action):
        prev_mask = self.mask
        self.mask = (self._apply_discrete(action) if self.action_type == 'discrete'
                     else self._apply_continuous(action))
        self.t   += 1
        dice_t, hd_t = self._snapshot_metrics()
        dice_p, _    = self.history[-1]
        reward       = dice_t - dice_p
        self.history.append((dice_t, hd_t))

        # Composite early stop
        done = self.t >= self.max_steps
        if not done and len(self.history) > self.stop_n:
            recent = self.history[-(self.stop_n + 1):]
            d_dices = [abs(recent[i+1][0] - recent[i][0]) for i in range(self.stop_n)]
            d_hds   = [abs(recent[i+1][1] - recent[i][1]) for i in range(self.stop_n)]
            if all(dd < self.stop_eps_dice for dd in d_dices) and \
               all(dh < self.stop_eps_hd   for dh in d_hds):
                done = True
        return self._state(), float(reward), bool(done), {'dice': dice_t, 'hd95': hd_t}

    # ── private ──────────────────────────────────────────────────────────────
    def _apply_discrete(self, a: int) -> np.ndarray:
        sp = self.shift_px
        if a == 0: return ndi.binary_dilation(self.mask, STRUCT).astype(np.uint8)
        if a == 1: return ndi.binary_erosion(self.mask, STRUCT).astype(np.uint8)
        if a == 2: return shifted(self.mask, -sp,   0)   # up
        if a == 3: return shifted(self.mask, +sp,   0)   # down
        if a == 4: return shifted(self.mask,   0, -sp)   # left
        if a == 5: return shifted(self.mask,   0, +sp)   # right
        if a == 6: return self.mask                       # no-op
        raise ValueError(f'Bad discrete action {a}')

    def _apply_continuous(self, a) -> np.ndarray:
        # a = (dy_frac, dx_frac) ∈ [-0.1, 0.1]
        dy = int(round(float(a[0]) * self.H))
        dx = int(round(float(a[1]) * self.W))
        return shifted(self.mask, dy, dx)

    def _snapshot_metrics(self):
        return dice_score(self.mask, self.gt), hd95_px(self.mask, self.gt)

    def _state(self) -> np.ndarray:
        sdt = (ndi.distance_transform_edt(self.mask) -
               ndi.distance_transform_edt(1 - self.mask))
        sdt = np.clip(sdt, -self.sdt_clip, self.sdt_clip) / self.sdt_clip
        return np.stack([
            self.image,
            self.mask.astype(np.float32),
            sdt.astype(np.float32),
            self.init_mask.astype(np.float32),
        ], axis=0)


env = SegmentationEnv(image, gt, init, max_steps=CFG['max_steps'],
                      shift_px=CFG['shift_px'], sdt_clip=CFG['sdt_clip'],
                      stop_eps_dice=CFG['early_stop_eps'],
                      stop_eps_hd=CFG['early_stop_hd'],
                      stop_n=CFG['early_stop_n'])
s0 = env.reset()
print(f'Initial state shape: {s0.shape}  (expect (4, {CFG["image_size"][0]}, {CFG["image_size"][1]}))')
print(f'Initial Dice: {env.history[0][0]:.4f}  HD95: {env.history[0][1]:.2f} px')

## 4 · State Visualisation (sanity #1)

In [ ]:
env.reset()
state = env._state()
names = ['ch0: image', 'ch1: current mask', 'ch2: SDT (norm)', 'ch3: U-Net init']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    cmap = 'gray' if i in (0, 1, 3) else 'RdBu_r'
    vmin, vmax = (None, None) if i in (0, 1, 3) else (-1, 1)
    im = ax.imshow(state[i], cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(names[i]); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle(f'Initial state — {sample["patient"]}_{sample["view"]}_{sample["phase"]} | '
             f'class={CFG["class_names"][CFG["target_class"]]}')
plt.tight_layout(); plt.show()

print('Sanity:')
print(f'  ch0 range: [{state[0].min():.3f}, {state[0].max():.3f}]   (expect [0, 1])')
print(f'  ch1 unique: {np.unique(state[1])}                          (expect [0., 1.])')
print(f'  ch2 range: [{state[2].min():.3f}, {state[2].max():.3f}]   (expect [-1, 1])')
print(f'  ch3 == ch1? {np.array_equal(state[1], state[3])}            (expect True at t=0)')

## 5 · Reward Sanity Test (sanity #2)
Apply known perturbations to a perfect mask and verify reward sign matches Dice direction.  
If any of these fail, something is wrong with `dice_score` or the env step loop.

In [ ]:
# Use GT as both init mask and GT → starts at perfect Dice = 1.0
# Then test that each action moves us in the expected direction.
test_env = SegmentationEnv(image, gt, gt, max_steps=20, shift_px=2)

# Sanity 1: starting state has Dice = 1.0
test_env.reset()
assert abs(test_env.history[0][0] - 1.0) < 1e-6, 'init Dice should be 1.0'

# Sanity 2: any non-no-op action from a perfect mask must give r ≤ 0
rewards = {}
for action_id, name in enumerate(['dilate','erode','↑','↓','←','→','no-op']):
    test_env.reset()
    _, r, _, info = test_env.step(action_id)
    rewards[name] = (r, info['dice'])

print(' Action  |  Reward  |  Dice After  |  Expected sign')
print('─────────────────────────────────────────────────────')
expected_sign = {'dilate':'≤0', 'erode':'≤0', '↑':'≤0', '↓':'≤0', '←':'≤0', '→':'≤0', 'no-op':'==0'}
for name, (r, d) in rewards.items():
    sign_ok = (abs(r) < 1e-6 if name == 'no-op' else r <= 1e-6)
    print(f'  {name:6s}  |  {r:+.4f} |   {d:.4f}    |   {expected_sign[name]:5s} {"✓" if sign_ok else "✗"}')

# Sanity 3: starting from U-Net init (sub-optimal), erode/dilate near a perfect boundary should hurt
real_env = SegmentationEnv(image, gt, init, max_steps=20, shift_px=2)
real_env.reset()
init_dice = real_env.history[0][0]
_, r, _, info = real_env.step(6)   # no-op
assert abs(r) < 1e-6, f'no-op reward should be 0, got {r}'
print(f'\nReal sample — init Dice: {init_dice:.4f}  |  no-op reward: {r:+.6f} (expect ≈0)')

# Sanity 4: cumulative reward = final Dice − initial Dice
real_env.reset()
cum_r, init_dice = 0.0, real_env.history[0][0]
for _ in range(10):
    _, r, _, info = real_env.step(np.random.randint(7))
    cum_r += r
drift = abs(cum_r - (info['dice'] - init_dice))
print(f'Cumulative reward sanity: |Σr − (Dice_final − Dice_init)| = {drift:.2e}  (expect <1e-6)')
assert drift < 1e-6, 'reward telescoping broken'

## 6 · Compute Benchmark — Random-Action Throughput
Times 1000 random env steps. Projects to Week 2 / 3 budgets.  
Note: this measures **env-only** cost. Add agent forward+backward time on top during real training.

In [ ]:
bench_env = SegmentationEnv(image, gt, init, max_steps=10**9, shift_px=2)
bench_env.reset()

# Warm-up
for _ in range(20):
    bench_env.step(np.random.randint(7))

# Timed run
N = CFG['benchmark_steps']
t0 = time.perf_counter()
for _ in range(N):
    bench_env.step(np.random.randint(7))
elapsed = time.perf_counter() - t0

steps_per_sec = N / elapsed
print(f'\n── Env Throughput ──────────────────────────────')
print(f'  {N} random steps in {elapsed:.2f}s = {steps_per_sec:.0f} steps/sec')
print(f'  Per-step cost: {1000*elapsed/N:.2f} ms')

In [ ]:
# Project to training budgets — env-only cost.
# Real training adds ~2-5ms agent forward+backward per step on T4.
AGENT_OVERHEAD_MS = 4.0  # rough estimate
effective_per_step_ms = 1000 / steps_per_sec + AGENT_OVERHEAD_MS

for budget_name, n_steps in [('Discrete agents', 50_000), ('DDPG', 100_000)]:
    secs_per_seed = n_steps * effective_per_step_ms / 1000
    hrs_per_seed  = secs_per_seed / 3600
    print(f'  {budget_name:20s} — {n_steps:6d} steps × 3 seeds = '
          f'{3*hrs_per_seed:.1f} h total at {effective_per_step_ms:.1f} ms/step')

n_discrete = 3 * 3 * 50_000 * effective_per_step_ms / 1000 / 3600   # 3 discrete agents × 3 seeds
n_ddpg     = 1 * 3 * 100_000 * effective_per_step_ms / 1000 / 3600  # DDPG × 3 seeds
n_msa      = 2 * 3 *  75_000 * effective_per_step_ms / 1000 / 3600  # 2 MSA agents × 3 seeds (MSA adds ~30%)
total      = n_discrete + n_ddpg + n_msa

print(f'\n── Projected Total ────────────────────────────')
print(f'  Week 2 (3 discrete × 3 seeds):  ~{n_discrete:.1f} h')
print(f'  Week 3 DDPG × 3 seeds:           ~{n_ddpg:.1f} h')
print(f'  Week 3 MSA × 2 × 3 seeds:        ~{n_msa:.1f} h')
print(f'  TOTAL train time:                 ~{total:.1f} h')
print(f'  Kaggle weekly GPU quota:          ~30 h')
print(f'  Decision: {"GO ✓" if total < 28 else "REVISE budgets — too tight"}')

## 7 · Mini Episode Visualisation
Run one 20-step random episode, plot Dice / HD95 trajectory.

In [ ]:
trace_env = SegmentationEnv(image, gt, init, max_steps=20, shift_px=2)
trace_env.reset()
actions, dices, hds, rewards = [], [trace_env.history[0][0]], [trace_env.history[0][1]], [0.0]

for _ in range(20):
    a = np.random.randint(7)
    _, r, done, info = trace_env.step(a)
    actions.append(a); dices.append(info['dice']); hds.append(info['hd95']); rewards.append(r)
    if done: break

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(dices, marker='o'); axes[0].axhline(dices[0], color='gray', ls='--', label='start')
axes[0].set(xlabel='step', ylabel='Dice', title=f'Dice trajectory (random policy)'); axes[0].legend()
axes[1].plot(hds,   marker='o', color='C1'); axes[1].set(xlabel='step', ylabel='HD95 (px)', title='HD95 trajectory')
axes[2].plot(rewards, marker='.', color='C2'); axes[2].axhline(0, color='k', lw=0.5)
axes[2].set(xlabel='step', ylabel='reward', title='Reward trajectory')
plt.tight_layout(); plt.show()

print(f'Episode length: {len(actions)} steps')
print(f'Init Dice: {dices[0]:.4f}  →  Final Dice: {dices[-1]:.4f}  (Δ = {dices[-1]-dices[0]:+.4f})')
print(f'Σ rewards  = {sum(rewards):+.4f}   (should equal Δ Dice above)')

## 8 · Decision Summary
If everything above passed:
- ✅ State shape locked at (4, 256, 256)
- ✅ Reward formula locked: `r_t = Dice(t) − Dice(t-1)`
- ✅ Compute budget verified or revised
- ✅ `SegmentationEnv` ready to be promoted to `iteris/env.py` paste-cell content

**Next:** `week2_01_random_baseline.ipynb` — random-action baseline across the test set.  
Then `week2_02_dqn_camus.ipynb` — first real agent.